In [1]:
from autogen_agentchat.agents import AssistantAgent
from autogen_ext.models.openai import OpenAIChatCompletionClient
from dotenv import load_dotenv
import os

In [2]:
load_dotenv()

True

In [3]:
model_client = OpenAIChatCompletionClient(model='gpt-4o-mini')

In [4]:
assitant = AssistantAgent(name='assitant', model_client=model_client, description='basic agent')

## another agent with tools and system message provided 

In [5]:
async def web_search(query: str) -> str:
    """find information from web"""
    return 'labrador dog is a breed of dog that is known for its friendly and outgoing personality. They are often used as family pets and are also popular as service dogs due to their intelligence and trainability.' 

In [6]:
agent = AssistantAgent(name='assitant',
                       model_client=model_client,
                       tools=[web_search],
                       system_message='use tools to solve the tasks',
                       description='an agent which uses tool to help solve task')

In [7]:
result = await agent.run(task='what is labrador')
print(result.messages[-1].content.strip())

labrador dog is a breed of dog that is known for its friendly and outgoing personality. They are often used as family pets and are also popular as service dogs due to their intelligence and trainability.


In [8]:
print((result.messages))

[TextMessage(source='user', models_usage=None, metadata={}, content='what is labrador', type='TextMessage'), ToolCallRequestEvent(source='assitant', models_usage=RequestUsage(prompt_tokens=61, completion_tokens=19), metadata={}, content=[FunctionCall(id='call_3wu1DRHhQNuP3FSh5ZSeunIf', arguments='{"query":"what is a labrador"}', name='web_search')], type='ToolCallRequestEvent'), ToolCallExecutionEvent(source='assitant', models_usage=None, metadata={}, content=[FunctionExecutionResult(content='labrador dog is a breed of dog that is known for its friendly and outgoing personality. They are often used as family pets and are also popular as service dogs due to their intelligence and trainability.', name='web_search', call_id='call_3wu1DRHhQNuP3FSh5ZSeunIf', is_error=False)], type='ToolCallExecutionEvent'), ToolCallSummaryMessage(source='assitant', models_usage=None, metadata={}, content='labrador dog is a breed of dog that is known for its friendly and outgoing personality. They are often 

## on_message() method

In [9]:
# %pip install autogen-core
from autogen_core import CancellationToken
# from autogen_core.messages import TextMessage
from autogen_agentchat.messages import TextMessage

async def assitant_run()-> None:
    response = await agent.on_messages(
        messages=[TextMessage(content = 'find info about labrador', 
                            source = 'user')],
        cancellation_token=CancellationToken())
    print(response)

await assitant_run()

Response(chat_message=ToolCallSummaryMessage(source='assitant', models_usage=None, metadata={}, content='labrador dog is a breed of dog that is known for its friendly and outgoing personality. They are often used as family pets and are also popular as service dogs due to their intelligence and trainability.', type='ToolCallSummaryMessage'), inner_messages=[ToolCallRequestEvent(source='assitant', models_usage=RequestUsage(prompt_tokens=140, completion_tokens=18), metadata={}, content=[FunctionCall(id='call_3x5NpvTlZ6k5Chj07kYd4oVW', arguments='{"query":"labrador breed information"}', name='web_search')], type='ToolCallRequestEvent'), ToolCallExecutionEvent(source='assitant', models_usage=None, metadata={}, content=[FunctionExecutionResult(content='labrador dog is a breed of dog that is known for its friendly and outgoing personality. They are often used as family pets and are also popular as service dogs due to their intelligence and trainability.', name='web_search', call_id='call_3x5N

# run stream message 

In [10]:
from autogen_agentchat.ui import Console

async def assitant_run_stream()-> None:
    await Console(
        agent.on_messages_stream(
            messages=[TextMessage(content='find about labrador dog', source='user')],
            cancellation_token=CancellationToken()
        ),output_stats=True
    )

await assitant_run_stream()    

---------- ToolCallRequestEvent (assitant) ----------
[FunctionCall(id='call_wn9SfncoczQjSBMSNWNKma0t', arguments='{"query":"labrador dog facts and information"}', name='web_search')]
[Prompt tokens: 218, Completion tokens: 20]
---------- ToolCallExecutionEvent (assitant) ----------
[FunctionExecutionResult(content='labrador dog is a breed of dog that is known for its friendly and outgoing personality. They are often used as family pets and are also popular as service dogs due to their intelligence and trainability.', name='web_search', call_id='call_wn9SfncoczQjSBMSNWNKma0t', is_error=False)]
---------- assitant ----------
labrador dog is a breed of dog that is known for its friendly and outgoing personality. They are often used as family pets and are also popular as service dogs due to their intelligence and trainability.
---------- Summary ----------
Number of inner messages: 2
Total prompt tokens: 218
Total completion tokens: 20
Duration: 1.70 seconds


In [11]:
from autogen_agentchat.ui import Console

async def assitant_run_stream()-> None:
    await Console(
        agent.on_messages_stream(
            messages=[TextMessage(content='what was the last question asked ', source='user')],
            cancellation_token=CancellationToken()
        ),output_stats=True
    )

await assitant_run_stream()    

---------- assitant ----------
The last question asked was: "find about labrador dog."
[Prompt tokens: 299, Completion tokens: 15]
---------- Summary ----------
Number of inner messages: 0
Total prompt tokens: 299
Total completion tokens: 15
Duration: 0.86 seconds


# ollama

In [14]:
from autogen_core.models import UserMessage
from autogen_ext.models.ollama import OllamaChatCompletionClient
from autogen_agentchat.agents import AssistantAgent

# ✅ Use llama3 (what you actually have installed)
ollama_model_client = OllamaChatCompletionClient(
    model="llama3")

# Quick test
response = await ollama_model_client.create([
    UserMessage(content="What is the capital of France?", source="user")
])
print(response)
await ollama_model_client.close()

finish_reason='stop' content='The capital of France is Paris.' usage=RequestUsage(prompt_tokens=17, completion_tokens=8) cached=False logprobs=None thought=None


In [15]:
from autogen_agentchat.agents import AssistantAgent
agent = AssistantAgent(
    name='assitant',
    model_client=ollama_model_client,
    system_message='you are helpful and aggresive assitant'
)

result = await agent.run(task='what is capital of hyderabad')
print(result.messages[-1].content)

I'M GLAD YOU ASKED THAT!

THE CAPITAL OF HYDERABAD IS HYDERABAD!


# openrouter

In [16]:
import asyncio
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.teams import SelectorGroupChat
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_agentchat.conditions import TextMentionTermination
from dotenv import load_dotenv
import os

load_dotenv()


True

In [ ]:
open_router_api_key='sk-or-v1-xxxx'


In [18]:
open_router_model_client =  OpenAIChatCompletionClient(
    base_url="https://openrouter.ai/api/v1",
    model="deepseek/deepseek-r1-0528:free",
    api_key = open_router_api_key,
    model_info={
        "family":'deepseek',
        "vision" :True,
        "function_calling":True,
        "json_output": False
    }
)

/Users/sivavenu/Desktop/AI_projects/autogen_projects/.venv/lib/python3.12/site-packages/autogen_ext/models/openai/_openai_client.py:413: UserWarning: Missing required field 'structured_output' in ModelInfo. This field will be required in a future version of AutoGen.
  validate_model_info(self._model_info)


In [19]:
assistant_agent1 = AssistantAgent(
    name = 'helpful_agent',
    model_client = open_router_model_client,
    system_message='You are a helpful assistant Agent'
)

result = await assistant_agent1.run(task = 'Hey how are you today ?')
print(result.messages)

NotFoundError: Error code: 404 - {'error': {'message': 'No endpoints found for deepseek/deepseek-r1-0528:free.', 'code': 404}, 'user_id': 'user_36LxqFxaDRqmWvV1gN8u1DvGW6T'}

# openrouter Free API example

In [24]:
from openai import OpenAI

client = OpenAI(
  base_url="https://openrouter.ai/api/v1",
  api_key=open_router_api_key,
)

# First API call with reasoning
response = client.chat.completions.create(
  model="stepfun/step-3.5-flash",
  messages=[
          {
            "role": "user",
            "content": "How many r's are in the word 'strawberry'?"
          }
        ],
  extra_body={"reasoning": {"enabled": True}}
)

# Extract the assistant message with reasoning_details
response = response.choices[0].message
print('response.....', response.content, '\n')
# Preserve the assistant message with reasoning_details
messages = [
  {"role": "user", "content": "How many r's are in the word 'strawberry'?"},
  {
    "role": "assistant",
    "content": response.content,
    "reasoning_details": response.reasoning_details  # Pass back unmodified
  },
  {"role": "user", "content": "Are you sure? Think carefully."}
]

# Second API call - model continues reasoning from where it left off
response2 = client.chat.completions.create(
  model="stepfun/step-3.5-flash",
  messages=messages,
  extra_body={"reasoning": {"enabled": True}}
)
print('response2.....', response2.choices[0].message.content)

response..... There are 3 r's in the word "strawberry". 

response2..... You're right to ask for careful checking. Let's spell it out:

**s t r a w b e r r y**

Positions of **r**:
- 3rd letter: **r**
- 8th letter: **r**
- 9th letter: **r**

That’s **3 r’s** in total.
